# End-to-End Large File Processing (Async)

This notebook demonstrates how to process large files (>15MB) using **Async Mode** for better concurrency and **Manual Polling** for progress transparency.

### Setting up the Latence client

In [1]:
import asyncio
import time
from pathlib import Path
from latence import AsyncLatence

client = AsyncLatence(
    api_key="lat_DhGuRk4jwbpfJbaerqB5Cp",
    base_url="https://latence-api-gateway.curly-wood-1df9.workers.dev",
    timeout=600,
)
print("Async client initialized.")

Async client initialized.


### Some helper functions

In [2]:
import tiktoken

def count_tokens(text, model="cl100k_base"):
    enc = tiktoken.get_encoding(model)
    return len(enc.encode(text))


### Parsing a large pdf to structured Markdown

In [ ]:
async def process_large_file(file_path: str, output_path: str = None):
    print(f"Submitting large file for processing: {file_path}")
    
    try:
        # 1. Submit the job
        # Files >10MB are uploaded directly to B2 via presigned URL (no base64)
        job = await client.document_intelligence.process(
            file_path=file_path,
            mode="performance",
            #use_layout_detection=False,
            pipeline_options={
                "use_ocr_for_image_block": True,
                "use_layout_detection": True,
                "use_chart_recognition": True,
                "use_seal_recognition": False,
                "use_doc_orientation_classify": True,
                "format_block_content": True,
                "merge_layout_blocks": True
            },
            return_job=True
        )
       
        print(f"Job submitted successfully! Job ID: {job.job_id}")
        
        # 2. Manual Polling for progress transparency
        print("Polling for job status...")
        start_time = time.time()
        
        while True:
            status_response = await client.jobs.get(job.job_id)
            elapsed = time.time() - start_time
            print(f"[{elapsed:4.1f}s] Status: {status_response.status}")
            
            if status_response.status in ("COMPLETED", "CACHED", "PULLED"):
                print(f"Processing finished in {elapsed:.1f}s!")
                break
            elif status_response.status == "FAILED":
                print(f"Job failed: {status_response.error_message}")
                return None
            
            await asyncio.sleep(5)
        
        # 3. Retrieve final output (automatically fetches from B2 for large results)
        print("Retrieving final extracted content...")
        result = await client.jobs.retrieve(job.job_id)
        
        content = result.get("content", "")
        print(f"Extracted {len(content)} characters.")
        
        # 4. Save as markdown
        if output_path is None:
            output_path = Path(file_path).stem + ".md"
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"Saved to {output_path}")
        
        print("\n--- CONTENT PREVIEW ---")
        print(content[:500] + "...")
        
        return content
        
    except Exception as e:
        print(f"An error occurred during process_large_file: {e}")
        return None

# Start the process
large_file_path = "/workspace/latence-python/notebooks/data/test_files/zollrecht.pdf"
processed_data = await process_large_file(large_file_path, output_path="/workspace/zollrecht.md")

Submitting large file for processing: /workspace/latence-python/notebooks/data/test_files/zollrecht.pdf


### Clean and chunk the parsed markdown data

In [ ]:
import sys
import os
import pickle

# Add paths if libraries are not installed in environment
repo_path = "/workspace/latency-index-3/src"
if repo_path not in sys.path:
    sys.path.append(repo_path)

sdk_path = "/workspace/latence-python/src"
if sdk_path not in sys.path:
    sys.path.append(sdk_path)

from preprocessing.chunking_engine import StatelessChunker, ChunkerConfig, DocumentFormat, SplittingStrategy
# Import the new utility from the SDK
from latence.resources.document_intelligence import clean_markdown

# Configuration
config = ChunkerConfig(
    chunk_size=1024,
    splitting_strategy=SplittingStrategy.TOKEN,
    document_format=DocumentFormat.MARKDOWN,
    chunk_overlap=64,
    remove_empty_chunks=True
)

# Load the file
input_path = "/workspace/zollrecht.md"
if os.path.exists(input_path):
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Optimization: Use the client SDK utility
    print("Cleaning text (removing images and layout divs)...")
    text = clean_markdown(text)

    # Process
    print(f"Chunking {input_path}...")
    chunks = StatelessChunker.chunk_text(text, config)
    print(f"Generated {len(chunks)} chunks.")

    # Save
    output_path = "/workspace/zollrecht_chunks.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(chunks, f)
    print(f"Saved chunks to {output_path}")

else:
    print(f"File not found: {input_path}")

Cleaning text (removing images and layout divs)...
Chunking /workspace/zollrecht.md...
Generated 333 chunks.
Saved chunks to /workspace/zollrecht_chunks.pkl


In [4]:
import random
from pprint import pprint
print(chunks[1].content)

[Context: I. Einführung in das Zollrecht > 1.2 Vertragliche Grundlagen/Rechtsgrundlagen]

#### 1.2 Vertragliche Grundlagen/Rechtsgrundlagen

Das wichtigste Handelsabkommen ist das seit 1947 bestehende Allgemeine Zoll- und Handelsabkommen (GATT – General Agreement on Tariffs and Trade). Es ist die Basis aller modernen Vorschriften und hat folgende Ziele definiert:

<table border="1" style="margin: auto; word-wrap: break-word;"><tr><td style="text-align: center; word-wrap: break-word;">Beseitigung von Diskriminierung im internationalen Handel</td></tr><tr><td style="text-align: center; word-wrap: break-word;">Förderung auf Gegenseitigkeit beruhender vorteilhafter Vereinbarungen (Reziprozität)</td></tr><tr><td style="text-align: center; word-wrap: break-word;">Förderung des weltweiten Handels</td></tr></table>

Auf diesem elementaren Vertrag basieren alle weiteren internationalen Zollabkommen der Nachkriegsgeschichte.

Im Jahr 1994 wurde aus dem GATT heraus in der Uruguay-Runde die Weltha

In [5]:
# Usage
print(count_tokens(chunks[3].content))

1014


### Extract entities and labels from chunks

In [ ]:
from latence import AsyncLatence
from latence._utils import process_batch_concurrently

# Assuming 'chunks' is your list of TextChunk objects

#example = [chunks[3]]

async def run_batch_extraction():
        # Define the single-item processor (closure captures client)
    async def _extract_one(text):
        return await client.extraction.extract(
            text=text,
            config={
                "label_mode": "generated",
                "threshold": 0.7,
                "enable_refinement": True
            },
            return_job=False
        )

    # Execute concurrent batch
    results = await process_batch_concurrently(
        items=[c.content for c in chunks],
        processor=_extract_one,
        max_concurrency=32
    )

    # Apply results
    # for chunk, result in zip(chunks, results):
    #     if isinstance(result, Exception):
    #         print(f"Failed: {result}")
    #     else:
    #         chunk.content = result.redacted_text
    
    return results

# In notebook:
extract_results = await run_batch_extraction()

# Save results
output_path = "/workspace/zollrecht_extracteed_chunks.pkl"
with open(output_path, "wb") as f:
    pickle.dump(extract_results, f)
print(f"Saved chunks to {output_path}")

Saved chunks to /workspace/zollrecht_extracteed_chunks.pkl


In [15]:
chunks[0].content

'--- Page 1 ---\n\n\n\n## I. Einführung in das Zollrecht\n\n\n\n## 1. Grundlagen des Zoll- und Außenwirtschaftsrechts\n\n\n\n#### 1.1 Bedeutung und historischer Wandel\n\nDas Zoll- und Außenwirtschaftsrecht ist ein maßgebendes Regulativ des internationalen Handelssystems.\n\nIn jedem Staat dieser Welt gibt es Zollvorschriften. Selbst Gebiete, die völkerrechtlich noch nicht als Staaten gelten, haben in ihrer Konstituierungsentwicklung Zollsysteme etabliert (z.B. Kosovo oder Westjordanland, Gazastreifen).\n\nMittelpunkt des traditionellen Zollrechts ist die Zollerhebung zu unterschiedlichen Zwecken. Im modernen Zollrecht stehen hingegen Sicherheitsaspekte im Vordergrund, die Zollerhebung verliert dabei mehr und mehr an Bedeutung.\n\nGrenzüberschreitende Geschäfte werden schon so lange getätigt, wie es Siedlungen mit Herrschafts- und Verwaltungsstrukturen gibt. Bereits im Altertum, u.a. in den Hochkulturen des alten Ägypten oder Mesopotamien, dienten Zolleinnahmen als wesentliche Pfeiler 

In [8]:
print(extract_results[0])

success=True request_id='req_590274041e2b816a' credits_used=None credits_remaining=None rate_limit_remaining=995 original_text='--- Page 1 ---\n\n\n\n## I. Einführung in das Zollrecht\n\n\n\n## 1. Grundlagen des Zoll- und Außenwirtschaftsrechts\n\n\n\n#### 1.1 Bedeutung und historischer Wandel\n\nDas Zoll- und Außenwirtschaftsrecht ist ein maßgebendes Regulativ des internationalen Handelssystems.\n\nIn jedem Staat dieser Welt gibt es Zollvorschriften. Selbst Gebiete, die völkerrechtlich noch nicht als Staaten gelten, haben in ihrer Konstituierungsentwicklung Zollsysteme etabliert (z.B. Kosovo oder Westjordanland, Gazastreifen).\n\nMittelpunkt des traditionellen Zollrechts ist die Zollerhebung zu unterschiedlichen Zwecken. Im modernen Zollrecht stehen hingegen Sicherheitsaspekte im Vordergrund, die Zollerhebung verliert dabei mehr und mehr an Bedeutung.\n\nGrenzüberschreitende Geschäfte werden schon so lange getätigt, wie es Siedlungen mit Herrschafts- und Verwaltungsstrukturen gibt. Be

### Redaction

In [16]:
import asyncio
import pickle
from latence import AsyncLatence
from latence._utils import process_batch_concurrently

# 1. Run Async Batch Redaction
async def run_redaction_and_save():
    print(f"Redacting {len(chunks)} chunks...")

    # Define processor
    async def _redact_one(text):
        return await client.redaction.detect_pii(
            text=text,
            config={"redact": True, "enable_refinement": True},  # <--- Add your config here
            return_job=False
        )

    # Run concurrently (max 64)
    raw_results = await process_batch_concurrently(
        items=[c.content for c in chunks],
        processor=_redact_one,
        max_concurrency=64
    )

    # Apply results to chunks
    success_count = 0
    failures = []

    final_results = []
    
    for i, res in enumerate(raw_results):
        if isinstance(res, Exception):
            print(f"Chunk {i} failed: {res}")
            failures.append((i, res))
        else:
            # Update chunk content with redacted text
            if res.redacted_text:
                final_results.append(res.redacted_text)
            else:
                # Fallback if specific redaction wasn't applied but detection ran
                # (only relevant if you set redact=False vs True)
                pass 
            success_count += 1

    print(f"Success: {success_count}/{len(chunks)}")

    # 2. Save Updated Chunks to Pickle
    # This saves the full Chunk objects with the new redacted content
    output_path = "/workspace/zollrecht_redacted_chunks.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(chunks, f)
    print(f"Saved chunks to {output_path}")

    # Optional: Save raw results if you want the full PII detection metadata
    # (Filter out exceptions first as they might not be picklable or useful here)
    valid_results = [r for r in raw_results if not isinstance(r, Exception)]
    with open("/workspace/redaction_raw_response.pkl", "wb") as f:
        pickle.dump(valid_results, f)
    
    return final_results

# Run
redact_results = await run_redaction_and_save()

Redacting 333 chunks...
Success: 333/333
Saved chunks to /workspace/zollrecht_redacted_chunks.pkl


In [20]:
import random
print(random.choice(redact_results))

[Context: Gegenüberstellung der Verarbeitungslisten zum Stichtag 01.04.2020 > IV. Zollwertrecht > 2.1.2 Der tatsächlich gezahlte oder zu zahlende Preis > Beispiel]

##### Beispiel

Firma X in [CITY] kauft von der Firma Y in Thailand einen Restposten Handtaschen. Der Listenverkaufspreis pro Handtasche beläuft sich auf 25 €. Weil es sich um ein Modell der letzten Saison handelt, gewährt die Firma Y einen Rabatt von 40 %, sodass der tatsächlich gezahlte oder zu zahlende Preis pro Handtasche somit 15 € beträgt.

Damit ein Preisnachlass anerkannt werden kann, ist es nicht zwingend erforderlich, dass dieser zum maßgebenden Zeitpunkt schon gewährt wird. Allerdings muss die Höhe der Preisermäßigung bereits feststehen und im Kaufvertrag verankert sein. Außerdem muss sich der Preisnachlass auf die einzuführende Ware beziehen. Zum Beispiel wird Ihnen als Käufer einer Ware bei der Abnahme einer Mindestmenge innerhalb eines bestimmten Zeitraumes durch den Verkäufer ein Mengenrabatt gewährt. Dieser 

In [ ]:
# Cleanup
await client.close()
print_info("Client closed.")